# 月データでムーンベースの場所を決めよう（データ分析と探究活動）

月の公開データを使って、**月面基地をどこに建てるか**を自分で決めます。
コードを書く必要はありません。セルを上から順に実行し、`# ★ここを変える` と書いてある
数字だけを書き換えて、結果をワークシートに記録していきます。

## あなたのミッションを1つ選ぶ（ワークシートに○）

| ミッション | 基地に必要なこと |
|---|---|
| ☀️ **太陽光発電基地** | 一年中よく日が当たること（発電したい） |
| 🧊 **氷採掘基地** | 氷がありそうなこと（＝ずっと日が当たらない「永久影」） |
| 🏠 **有人基地（人が住む）** | 電力が使えて、かつ氷（永久影）のすぐ近くにあること |

同じデータでも、ミッションが変われば「最適な場所」は変わります。

## 進め方
1. 上から順にセルを実行する（Colab なら「ランタイム」→「すべてのセルを実行」）
2. 各ステップで、まずワークシートに **予想** を書く
3. `# ★ここを変える` の数字を書き換えて実行し、結果をワークシートに記録
4. **気づいたこと** を書く

In [ ]:
# 準備：ヘルパー（moonkit）を読み込む
from moonkit import *
for k in DATASETS:
    print(k, ':', len(load(k)), '地点')

---
## ステップ1：月の温度は1日でどれくらい変わる？

月には空気（大気）がありません。空気がないと、温度はどうなるでしょう？

**予想をワークシートに書いてから**、下のセルを実行します。

In [ ]:
my_band = (-10, 10)      # ★ここを変える：調べたい緯度の範囲（例：赤道なら (-10, 10)、緯度45度なら (40, 50)）

band = region(load('温度'), lat=my_band)
diurnal_curve(band)                       # 1日の温度変化カーブ
band = daily_swing(band)                  # 各地点の「1日の平均・較差・ばらつき」を計算
summary(band, 't_mean_K', 't_swing_K', 't_std_K')
#   t_mean_K  … 1日の平均温度
#   t_swing_K … 1日の温度差（いちばん暑い時 − いちばん寒い時）
#   t_std_K   … 温度のばらつき（分散の平方根）。カーブが大きく上下するほど大きい

**ワークシートに記録**：1日の温度差（`t_swing_K` の平均）は何 K？　地球の砂漠の昼夜差はせいぜい 20〜30 ℃ です。

**気づいたこと**：なぜこんなに差が大きいの？　カーブの形（朝の上がり方と、夕方〜夜の下がり方）は左右対称？

**もうひとつ試す**：`my_band` を `(-90, -80)`（南極のあたり）にして実行してみよう。
カーブがガタガタになり、温度差も小さくなる。極では太陽が地平線の近くを回るだけで
「昼」と「夜」がはっきりせず、**この温度データ自体も不確かになる**。
でも、その「昼夜がない」ことこそが、極を基地に向いた場所にしている。

---
## ステップ2：月の「海」と「陸」で何が違う？

月を見ると、黒っぽく平らな「海（マリア）」と、白っぽくでこぼこの「陸（高地）」があります。
まず、クレーターの分布から「海」を自分で見つけます。

**予想**：クレーターの数は、月面のどこでも同じくらい？　それとも場所によってかたよる？

In [ ]:
# (1) 全体のクレーターの密度を地図で見る → まわりより少ない「帯」を探す
grid_count(load('クレーター'), lat_step=10, lon_step=10)

In [ ]:
# (2) クレーターの位置を月面画像に重ねる → 少ない場所は、画像のどこ？
scatter(load('クレーター'), 'lon', 'lat')

In [ ]:
# (3) 公式の「海」の座標（USGS 地名辞典の23の海）を使って、海と陸のクレーターを数値で比べる
my_scale = 0.8      # ★ここを変える：海の範囲。1.0 で海の縁まで、小さくすると中心部だけ

c = near_maria(load('クレーター'), scale=my_scale)
print(c['区分'].value_counts())               # 海・陸それぞれのクレーターの数
print(summary_by(c, group='区分', value='diam_km'))   # 直径の平均など

# 年代（DeepCraters）でも比べてみる（数字が大きいほど新しい）
age = near_maria(load('クレーター年代'), scale=my_scale)
print('推定年代の平均：', age.groupby('区分')['Age'].mean().round(2).to_dict())

**ワークシートに記録**：海と陸のクレーターの数、直径の平均、年代の平均。

**気づいたこと**：
- 海のほうがクレーターが少ないのはなぜ？（ヒント：海は昔、溶けた溶岩でおおわれて古いクレーターが消えた）
- クレーターが少ない＝新しい、と言えるのはなぜ？

**基地との関係**：海は平らで**着陸しやすい**が、若い。極域の高地は古くてでこぼこだが、
**氷（永久影）がある**。基地はどちらを取る？　この「トレードオフ」をステップ4・5で考えます。

---
## ステップ3：月の南極を細かく見る

ステップ1で、月の昼夜の温度差はすさまじいこと、そして極では「昼夜」が成立せず、
Diviner の温度データも不確かになることが分かりました。
基地の候補として、**南極**をくわしく見ます。ここからは、極でも信頼できる
**日照（LOLA）のデータ** を使います。南極には「一年中日が当たらない場所（永久影）」と
「ほぼずっと日が当たる場所」が、となり合って存在します。

**予想**：南極の「平均日照率」のヒストグラムは、どんな形になると思う？

In [ ]:
極 = south_pole(load('極域日照'))          # 南緯80度より南
my_threshold = 5                          # ★ここを変える：「これより日照率が低ければ永久影だろう」というしきい値 [%]

hist(極, 'average_illumination_percent', vline=my_threshold)
暗い場所 = 極[極['average_illumination_percent'] <= my_threshold]
print('しきい値', my_threshold, '% 以下の地点：', len(暗い場所), '個')
scatter(暗い場所, 'lon', 'lat')           # その場所を地図に表示

**ワークシートに記録**：自分で決めたしきい値と、それに当てはまった地点の数。

**気づいたこと**：しきい値を大きく（ゆるく）すると、地点の数はどう変わる？

---
## ステップ4：あなたのミッションの基地はどこ？

複数の条件を「0〜1の点数」に直して重みをつけて合計し、点数の高い場所を探します。
**自分のミッションに合わせて `want` を書き換えます。**

| ミッション | おすすめの `want` |
|---|---|
| ☀️ 太陽光発電基地 | `{'average_illumination_percent': ('高い', 1)}` |
| 🧊 氷採掘基地 | `{'permanent_shadow_fraction': ('高い', 1)}` |
| 🏠 有人基地 | `{'average_illumination_percent': ('高い', 1), 'km_to_shadow': ('低い', 1), 'permanent_shadow_fraction': ('低い', 1)}` |

- `km_to_shadow` … いちばん近い永久影（＝氷がありそうな場所）までの距離。
- 有人基地は「**日向の尾根で発電しつつ、氷のすぐ隣**」をねらう。

重み（かっこ内の数字）を変えると、どの条件を重く見るかを調整できます。

In [ ]:
# 南極の各地点に「いちばん近い永久影までの距離（km_to_shadow）」を付ける
極 = dist_to_permanent_shadow(south_pole(load('極域日照')))

want = {                                              # ★ここを変える：上の表から自分のミッションのものを写す
    'average_illumination_percent': ('高い', 1),
    'km_to_shadow':                 ('低い', 1),
    'permanent_shadow_fraction':    ('低い', 1),
}

best = site_score(極, want, top=10)
print('あなたのミッションでの上位10地点：')
print(best[['lat', 'lon', 'average_illumination_percent',
            'permanent_shadow_fraction', 'km_to_shadow', 'スコア']].round(2).to_string(index=False))

# 全地点をスコアで色分けした地図（明るいほど条件に合う）
scatter(site_score(極, want, top=None), 'lon', 'lat', color='スコア')

**ワークシートに記録**：上位に出た場所の緯度・経度、そこの日照率・永久影率・氷までの距離。
そして「**ここに基地を建てる**」と決めた1地点と、その**理由（3つ以上の文で）**。

---
## ステップ5：他の班と比べる

- 他の班（ちがうミッション）が選んだ場所は、あなたの場所とどれくらい離れている？
- なぜ違う場所になった？　同じデータを使っているのに。
- 「1つの正解」はある？　それとも「目的によって最適地は変わる」？
- ステップ2 の「平らな海」と「氷のある極」のトレードオフは、どう効いた？

ここはコードなし。ワークシートに書いて、クラスで共有します。

---
## （発展・任意）ステップ6：機械が決めた基準と、自分が決めた基準を比べる

ステップ4 で、あなたは重みを決めて「有人基地に向く場所」を選びました。
その「向き・不向き」を、機械学習に**当てさせる**とどうなるでしょう。決定木・
ニューラルネットなど5種類のモデルを取り替えて、境界線の形・当たりやすさ・
「ルールを説明できるか」を比べます。

このステップは別ノートブック `course_moonbase_ml.ipynb` で行います。時間が
余った班・興味のある人向けで、必須ではありません。